# Demo: FP-Growth tự cài đặt bằng Julia

Notebook chạy trên **Google Colab** (kernel Python); Julia được tải về và gọi qua shell.
Chạy lần lượt từ trên xuống (Runtime → Run all).

> **Việc cần làm trước khi nộp:** thay `REPO_URL` bên dưới bằng link GitHub public của nhóm.

In [ ]:
REPO_URL = "https://github.com/hoangfish/fpgrowth"   # <-- THAY BẰNG REPO CỦA NHÓM
REPO_DIR = "fpgrowth-julia"

## 1. Cài Julia và lấy mã nguồn

In [ ]:
!curl -fsSL https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.4-linux-x86_64.tar.gz -o julia.tar.gz
!tar -xzf julia.tar.gz -C /usr --strip-components 1
!julia --version
!git clone {REPO_URL} {REPO_DIR} 2>/dev/null || (cd {REPO_DIR} && git pull)
%cd {REPO_DIR}

## 2. Kiểm thử tự động (phải pass toàn bộ)

In [ ]:
!julia --project test/runtests.jl

## 3. Ví dụ nhỏ (SPMF `contextPasquier99`, minsup = 40%)
Kết quả theo định dạng SPMF: `item1 item2 ... #SUP: support`.

In [ ]:
!cat data/toy/contextPasquier99.txt
!julia --project demo.jl --minsup 40% --output /tmp/toy_out.txt data/toy/contextPasquier99.txt
!sort /tmp/toy_out.txt

## 4. Chuẩn bị dữ liệu benchmark
Tải bộ dữ liệu thật từ FIMI (cùng định dạng với SPMF: mỗi dòng một giao dịch, item cách nhau bởi dấu cách).
Nếu một link lỗi, tải thủ công từ trang SPMF Datasets/FIMI hoặc từ bản host trên Hugging Face/Kaggle của nhóm,
rồi đặt vào `data/benchmark/` với tên `chess.txt`, `mushroom.txt`, `retail.txt`, `T10I4D100K.txt`.
Nếu không có dữ liệu thật, `gen_data.jl` sinh bản tổng hợp `*_like.txt` có thống kê tương tự.

In [ ]:
!python3 scripts/download_data.py


In [ ]:
!julia --project scripts/gen_data.jl        # chỉ cần nếu thiếu dữ liệu thật (sinh *_like.txt)
!julia --project scripts/gen_data.jl avglen # CSDL tổng hợp có độ dài giao dịch tăng dần

## 5. Chạy FP-Growth trên benchmark (so sánh bản cơ bản và bản tối ưu)

In [ ]:
!julia --project demo.jl --algo basic --minsup 0.2 data/benchmark/mushroom.txt
!julia --project demo.jl --algo opt   --minsup 0.2 data/benchmark/mushroom.txt

## 5b. Đối chiếu với SPMF (tuỳ chọn)
Tải `spmf.jar` rồi chạy `scripts/run_spmf.py`; script ghi `data/spmf_expected/` và `results/spmf_times.csv`.


In [ ]:
%%bash
[ -f spmf.jar ] || wget -q https://www.philippe-fournier-viger.com/spmf/spmf.jar -O spmf.jar
java -version 2>&1 | head -1
ls -la spmf.jar


In [ ]:
!python3 scripts/run_spmf.py


## 6. Toàn bộ thực nghiệm Chương 2
Mất vài phút đến vài chục phút tùy dữ liệu. Kết quả ghi vào `results/*.csv`, biểu đồ vào `docs/figures/`.
Nếu có `spmf.jar`, chạy thêm `scripts/run_spmf.py` trước để so sánh trực tiếp với SPMF.

In [ ]:
# Chạy lại toàn bộ thực nghiệm mất khoảng 1 giờ; bỏ dấu # ở dòng dưới nếu muốn chạy lại.
# Mặc định dùng kết quả đã có sẵn trong results/ (sinh bởi cùng script) để vẽ lại biểu đồ.
# !python3 scripts/run_experiments.py
!python3 scripts/plot_results.py


In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("docs/figures/*.png")):
    print(f); display(Image(f))

## 7. Chương 3: hai ứng dụng thực tế
Mọi kết quả do `fpgrowth` của nhóm sinh ra (không dùng thư viện FIM). Chuyển đổi dữ liệu thành giao dịch nằm trong `src/apps.jl` (`load_basket_csv`, `encode_sequence`, `load_splice`); luật kết hợp, tập đóng/tối đại, mẫu nổi bật trong `src/postprocess.jl`.

In [ ]:
!python3 scripts/download_app_data.py


### 7.1 Ứng dụng 1: phân tích giỏ hàng
**Quyết định biến đổi dữ liệu.** Mỗi giỏ là một giao dịch, mỗi mặt hàng là một item (bỏ trùng; không xét số lượng vì FP-Growth chỉ cần có/không). Retail dùng mã số ẩn danh, Groceries có tên mặt hàng nên dùng để diễn giải ý nghĩa kinh doanh.

In [ ]:
!julia --project scripts/app1_market_basket.jl retail


In [ ]:
!julia --project scripts/app1_market_basket.jl groceries


### 7.2 Ứng dụng 2: dữ liệu sinh học (splice-junction)
**Quyết định biến đổi dữ liệu.** Mỗi chuỗi DNA 60 nucleotide là một giao dịch gồm 60 item `(vị trí, nucleotide)`, mã hoá `(p-1)*4 + b`. Mã hoá theo vị trí giữ thông tin vị trí (nếu chỉ dùng tập nucleotide thì mọi chuỗi gần như giống nhau). Ký tự mơ hồ (N, D, S, R) bị bỏ qua.

In [ ]:
!julia --project scripts/app2_splice.jl


### 7.3 Biểu đồ và bảng kết quả

In [ ]:
!python3 scripts/plot_apps.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("docs/figures/app*.png")):
    print(f); display(Image(f))


In [ ]:
import pandas as pd
for f in ["app1_retail_top10.csv","app1_groceries_top10.csv","app1_groceries_top10_single.csv","app2_emerging.csv","app2_class_rules.csv","app2_eval.csv"]:
    try:
        print("==", f); display(pd.read_csv("results/apps/"+f).head(15))
    except Exception as e:
        print("thiếu", f, e)
